In [1]:
# 🟩 Cell 1: Install dependencies
!pip install ollama matplotlib pandas seaborn fpdf2 scikit-learn numpy tqdm retrying astunparse pylint
!pip install --upgrade ollama

In [2]:
# Add to your imports cell if missing:
from typing import Dict, Optional
from difflib import SequenceMatcher

In [3]:
# 🟩 Cell 2: Import all required modules
import os
import ollama
import subprocess
import difflib
import csv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fpdf import FPDF
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import json
import re
from datetime import datetime
import warnings
import concurrent.futures
from tqdm import tqdm
from retrying import retry
import ast
import astunparse
import pylint.lint
from typing import Dict, List, Optional, Tuple, Any
import hashlib

warnings.filterwarnings('ignore')

C:\Users\Rahul\Downloads\Anaconda\Projects\Lib\site-packages\fpdf\__init__.py:40: UserWarning: You have both PyFPDF & fpdf2 installed. Both packages cannot be installed at the same time as they share the same module namespace. To only keep fpdf2, run: pip uninstall --yes pypdf && pip install --upgrade fpdf2
  warnings.warn(


In [4]:
# 🟩 Cell 3: Set Windows paths for QuixBugs dataset
BASE_PATH = Path(r"C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master")
BUGGY_DIR = BASE_PATH / "python_programs"
CORRECT_DIR = BASE_PATH / "correct_python_programs" 
TEST_DIR = BASE_PATH / "python_testcases"
FIXED_DIR = BASE_PATH / "fixed_programs"
CACHE_DIR = BASE_PATH / "cache"
FIXED_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

# Log files
LOG_FILE = "repair_detailed_log.csv"
METRICS_FILE = "repair_metrics.json"
FAILURE_ANALYSIS_FILE = "failure_analysis.json"

# Enhanced LLM models list
LLM_MODELS = [
    "codellama:7b-instruct",
    "codellama:13b-instruct",
    "llama2:13b"
]

print(f"✅ Base Path: {BASE_PATH}")
print(f"✅ Buggy Programs: {BUGGY_DIR}")
print(f"✅ Correct Programs: {CORRECT_DIR}")
print(f"✅ Test Cases: {TEST_DIR}")
print(f"✅ Fixed Programs: {FIXED_DIR}")
print(f"✅ Cache Directory: {CACHE_DIR}")

✅ Base Path: C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master
✅ Buggy Programs: C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master\python_programs
✅ Correct Programs: C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master\correct_python_programs
✅ Test Cases: C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master\python_testcases
✅ Fixed Programs: C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master\fixed_programs
✅ Cache Directory: C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master\cache


In [5]:
# 🟩 Cell 4: Enhanced 14 Defect Categories with AST patterns
DEFECT_CATEGORIES = {
    'incorrect_assignment_operator': {
        'patterns': ['=', '+=', '-=', '*=', '/='],
        'ast_type': [ast.Assign, ast.AugAssign]
    },
    'incorrect_variable': {
        'patterns': ['variable name', 'wrong variable'],
        'ast_type': [ast.Name]
    },
    'incorrect_comparison_operator': {
        'patterns': ['==', '!=', '<', '>', '<=', '>='],
        'ast_type': [ast.Compare]
    },
    'missing_condition': {
        'patterns': ['if', 'elif', 'while condition'],
        'ast_type': [ast.If, ast.While]
    },
    'missing_added_plus_one': {
        'patterns': ['+1', '-1', 'off-by-one'],
        'ast_type': [ast.BinOp, ast.For]
    },
    'variable_swap': {
        'patterns': ['swap', 'reversed order'],
        'ast_type': [ast.Assign]
    },
    'incorrect_array_slice': {
        'patterns': ['[:', ':]', 'slice'],
        'ast_type': [ast.Subscript]
    },
    'variable_prepend': {
        'patterns': ['prepend', 'insert'],
        'ast_type': [ast.Call]
    },
    'incorrect_data_structure_constant': {
        'patterns': ['[]', '{}', 'None', '0'],
        'ast_type': [ast.List, ast.Dict, ast.Constant]
    },
    'incorrect_method_called': {
        'patterns': ['method call', 'function call'],
        'ast_type': [ast.Call]
    },
    'incorrect_field_dereference': {
        'patterns': ['.', 'attribute access'],
        'ast_type': [ast.Attribute]
    },
    'missing_arithmetic_expression': {
        'patterns': ['+', '-', '*', '/', '%'],
        'ast_type': [ast.BinOp]
    },
    'missing_function_call': {
        'patterns': ['function call missing'],
        'ast_type': [ast.Call]
    },
    'missing_line': {
        'patterns': ['missing statement', 'missing line'],
        'ast_type': [ast.Expr, ast.Assign, ast.Return]
    }
}

In [6]:
   import os
import re

def find_and_fix_tester_py_files(base_path):
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file == "tester.py":
                full_path = os.path.join(root, file)
                print(f"🔍 Checking: {full_path}")
                try:
                    with open(full_path, "r", encoding="utf-8") as f:
                        code = f.read()
                    compile(code, full_path, "exec")  # check for syntax
                except SyntaxError as e:
                    print(f"❌ Syntax error in: {full_path}\n   ↪ {e}")
                    
                    fixed_code = fix_syntax(code)
                    if fixed_code:
                        try:
                            compile(fixed_code, full_path, "exec")
                            with open(full_path, "w", encoding="utf-8") as f:
                                f.write(fixed_code)
                            print("✅ Auto-fixed!")
                        except SyntaxError:
                            print("⚠️ Fix attempt failed. Manual review needed.")
                    else:
                        print("⚠️ No fix attempt made.")
                except Exception as e:
                    print(f"⚠️ Unexpected error: {e}")
    print("✅ Done scanning and fixing.")

def fix_syntax(code):
    # Basic auto-fix: add colons where obviously missing
    lines = code.splitlines()
    new_lines = []
    for line in lines:
        stripped = line.strip()
        if re.match(r"^(def|if|for|while|elif|else)\b", stripped) and not stripped.endswith(":"):
            line += ":"
        new_lines.append(line)
    return "\n".join(new_lines)

# ✅ Run this on your dataset folder
find_and_fix_tester_py_files(r"C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master")


🔍 Checking: C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master\tester.py
✅ Done scanning and fixing.


In [7]:
# 🟩 Cell 5: Enhanced helper functions with caching and AST analysis
def get_file_hash(filepath: Path) -> str:
    """Generate MD5 hash of file contents for caching"""
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def load_code(filepath: Path) -> str:
    """Load code from file with error handling and caching"""
    try:
        with open(filepath, "r", encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"❌ Error loading {filepath}: {e}")
        return ""

def extract_code_from_response(response_text: str) -> str:
    """Enhanced code extraction with better cleaning"""
    # Remove markdown code blocks
    if "```python" in response_text:
        start = response_text.find("```python") + 9
        end = response_text.find("```", start)
        if end != -1:
            response_text = response_text[start:end].strip()
    elif "```" in response_text:
        start = response_text.find("```") + 3
        end = response_text.find("```", start)
        if end != -1:
            response_text = response_text[start:end].strip()
    
    # Clean up common LLM artifacts
    cleaned = response_text.strip()
    cleaned = re.sub(r'^Here.*?code:\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'^The.*?fix.*?:\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'^```.*$', '', cleaned, flags=re.MULTILINE)
    
    return cleaned

def analyze_code_with_ast(code: str) -> Dict:
    """Perform AST analysis to detect potential issues"""
    try:
        tree = ast.parse(code)
        issues = []
        
        # Check for common issues
        for node in ast.walk(tree):
            # Check for empty except blocks
            if isinstance(node, ast.ExceptHandler) and not node.body:
                issues.append("Empty except block")
            
            # Check for comparison with literals in if conditions
            if isinstance(node, ast.Compare) and any(isinstance(n, ast.Num) for n in node.comparators):
                issues.append("Comparison with literal")
            
            # Check for unused variables
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id.startswith('_'):
                        issues.append(f"Potentially unused variable: {target.id}")
        
        return {
            'ast_valid': True,
            'issues': issues,
            'ast_tree': astunparse.unparse(tree) if tree else None
        }
    except SyntaxError as e:
        return {
            'ast_valid': False,
            'error': str(e),
            'issues': ["Syntax error in code"]
        }

def run_pylint_analysis(code: str, filename: str) -> Dict:
    """Run pylint analysis on the code"""
    try:
        pylint_output = []
        
        def pylint_callback(result):
            pylint_output.append(result)
        
        # Create a temporary file for pylint analysis
        temp_file = CACHE_DIR / f"temp_{filename}"
        with open(temp_file, "w", encoding="utf-8") as f:
            f.write(code)
        
        # Run pylint
        pylint.lint.Run(
            [str(temp_file)],
            do_exit=False,
            reporter=pylint_callback
        )
        
        # Clean up
        temp_file.unlink()
        
        return {
            'pylint_issues': pylint_output,
            'pylint_score': sum(1 for issue in pylint_output if issue.category == 'error')
        }
    except Exception as e:
        return {
            'pylint_error': str(e)
        }

@retry(stop_max_attempt_number=3, wait_fixed=2000)
def run_test_with_validation(program_name: str) -> Tuple[str, int, bool]:
    """Run test with proper error handling, validation, and retries"""
    try:
        # Change to base directory for proper import paths
        original_dir = os.getcwd()
        os.chdir(BASE_PATH)


        # Run the tester with timeout
        result = subprocess.run(
            ["python", "tester.py", program_name], 
            capture_output=True, 
            text=True,
            timeout=30,
            cwd=str(BASE_PATH)
        )
        
        os.chdir(original_dir)
        
        # Parse output for pass/fail
        output = result.stdout + result.stderr
        passed = result.returncode == 0 and "PASSED" in output.upper()
        
        return output, 0 if passed else 1, passed
        
    except subprocess.TimeoutExpired:
        os.chdir(original_dir)
        return "Test timeout after 30 seconds", 1, False
    except Exception as e:
        os.chdir(original_dir)
        return f"Test execution error: {str(e)}", 1, False

In [8]:
# 🟩 Cell 6: Advanced defect categorization with AST and diff analysis
def categorize_defect_advanced(original_code: str, fixed_code: str, program_name: str = "") -> str:
    """Enhanced defect categorization using AST and diff analysis"""
    # First try AST-based analysis
    ast_result = analyze_defect_with_ast(original_code, fixed_code)
    if ast_result and ast_result != 'unknown_defect':
        return ast_result
    
    # Fall back to diff analysis if AST doesn't find anything
    return categorize_with_diff(original_code, fixed_code, program_name)

def analyze_defect_with_ast(original_code: str, fixed_code: str) -> Optional[str]:
    """Analyze defects using AST comparison"""
    try:
        original_ast = ast.parse(original_code)
        fixed_ast = ast.parse(fixed_code)
        
        # Compare AST nodes
        for orig_node, fixed_node in zip(ast.walk(original_ast), ast.walk(fixed_ast)):
            if type(orig_node) != type(fixed_node):
                continue
                
            # Check for comparison operator changes
            if isinstance(orig_node, ast.Compare) and isinstance(fixed_node, ast.Compare):
                if orig_node.ops != fixed_node.ops:
                    return "incorrect_comparison_operator"
                    
            # Check for assignment changes
            if isinstance(orig_node, ast.Assign) and isinstance(fixed_node, ast.Assign):
                if len(orig_node.targets) == len(fixed_node.targets) == 2:
                    if (orig_node.targets[0].id == fixed_node.targets[1].id and 
                        orig_node.targets[1].id == fixed_node.targets[0].id):
                        return "variable_swap"
                        
            # Check for operator changes
            if isinstance(orig_node, ast.BinOp) and isinstance(fixed_node, ast.BinOp):
                if type(orig_node.op) != type(fixed_node.op):
                    return "incorrect_assignment_operator"
                    
            # Check for function call changes
            if isinstance(orig_node, ast.Call) and isinstance(fixed_node, ast.Call):
                if orig_node.func.id != fixed_node.func.id:
                    return "incorrect_method_called"
                    
    except Exception:
        pass
        
    return None

def categorize_with_diff(original_code: str, fixed_code: str, program_name: str) -> str:
    """Categorize defects using diff analysis"""
    original_lines = original_code.splitlines()
    fixed_lines = fixed_code.splitlines()
    
    # Generate unified diff
    diff = list(difflib.unified_diff(original_lines, fixed_lines, lineterm=''))
    
    # Extract changed lines
    removed_lines = [line[1:] for line in diff if line.startswith('-') and not line.startswith('---')]
    added_lines = [line[1:] for line in diff if line.startswith('+') and not line.startswith('+++')]
    
    if not removed_lines and not added_lines:
        return "no_change"
    
    # Combine all change context
    change_context = ' '.join(removed_lines + added_lines).lower()
    
    # Enhanced pattern matching with priority ordering
    defect_checks = [
        ('missing_added_plus_one', lambda: any(p in change_context for p in ['range(', 'len(', '+1', '-1']) and 
                                          any(op in change_context for op in ['<', '>', '<=', '>='])),
        ('incorrect_comparison_operator', lambda: any(old_op in removed_lines[0] if removed_lines else '' and 
                                                   new_op in added_lines[0] if added_lines else '' 
                                                   for old_op in ['==', '!=', '<', '>', '<=', '>='] 
                                                   for new_op in ['==', '!=', '<', '>', '<=', '>='])),
        ('variable_swap', lambda: len(removed_lines) == 1 and len(added_lines) == 1 and
                                len(re.findall(r'\b[a-zA-Z_]\w*\b', removed_lines[0])) >= 2 and
                                len(re.findall(r'\b[a-zA-Z_]\w*\b', added_lines[0])) >= 2 and
                                set(re.findall(r'\b[a-zA-Z_]\w*\b', removed_lines[0])) == 
                                set(re.findall(r'\b[a-zA-Z_]\w*\b', added_lines[0]))),
        ('incorrect_assignment_operator', lambda: any(op in change_context for op in ['+=', '-=', '*=', '/=', '='])),
        ('incorrect_array_slice', lambda: '[' in change_context and (':' in change_context or ']' in change_context)),
        ('incorrect_method_called', lambda: '(' in change_context and ')' in change_context and
                                          any(method in change_context for method in ['append', 'pop', 'insert', 'remove'])),
        ('missing_function_call', lambda: '(' in change_context and ')' in change_context),
        ('missing_condition', lambda: any(keyword in change_context for keyword in ['if', 'elif', 'while', 'for'])),
        ('missing_line', lambda: 'return' in change_context),
        ('incorrect_data_structure_constant', lambda: any(const in change_context for const in ['[]', '{}', 'none', 'null', '0'])),
        ('incorrect_variable', lambda: len(set(re.findall(r'\b[a-zA-Z_]\w*\b', change_context))) > 1)
    ]
    
    for defect_type, check_fn in defect_checks:
        if check_fn():
            return defect_type
    
    return "unknown_defect"

In [9]:
# 🟩 Cell 7: Enhanced LLM interaction with retries, caching, and model fallback
@retry(stop_max_attempt_number=3, wait_exponential_multiplier=1000, wait_exponential_max=10000)
def query_llm_with_retry(model: str, prompt: str, timeout: int = 120) -> str:
    """Query LLM with retries and timeout"""
    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": 0.3,  # Slightly higher for creativity
                "top_p": 0.9,
                "timeout": timeout
            }
        )
        return response['message']['content']
    except Exception as e:
        print(f"❌ Error querying {model}: {e}")
        raise

def get_cached_response(cache_key: str) -> Optional[str]:
    """Check for cached response"""
    cache_file = CACHE_DIR / f"{cache_key}.json"
    if cache_file.exists():
        with open(cache_file, "r") as f:
            return json.load(f).get('response')
    return None

def cache_response(cache_key: str, response: str) -> None:
    """Cache the LLM response"""
    cache_file = CACHE_DIR / f"{cache_key}.json"
    with open(cache_file, "w") as f:
        json.dump({'response': response}, f)

def generate_cache_key(code: str, program_name: str, model: str) -> str:
    """Generate unique cache key"""
    return hashlib.md5((code + program_name + model).encode()).hexdigest()

def fix_code_with_ollama(code: str, program_name: str = "", models: List[str] = LLM_MODELS) -> Tuple[str, str]:
    """Enhanced code repair with caching and model fallback"""
    best_fix = code
    best_model = "none"
    last_error = None
    
    for model in models:
        cache_key = generate_cache_key(code, program_name, model)
        cached_response = get_cached_response(cache_key)
        
        if cached_response:
            print(f"  🔄 Using cached response for {model}")
            fixed_code = extract_code_from_response(cached_response)
            if is_valid_python(fixed_code):
                return fixed_code, f"{model} (cached)"
        
        try:
            print(f"  🔄 Trying model: {model}")
            prompt = generate_repair_prompt(code, program_name)
            
            response = query_llm_with_retry(model, prompt)
            cache_response(cache_key, response)
            
            fixed_code = extract_code_from_response(response)
            
            # Validate the fix before accepting
            if is_valid_python(fixed_code):
                best_fix = fixed_code
                best_model = model
                break
                
        except Exception as e:
            last_error = e
            continue
    
    if best_fix == code and last_error:
        print(f"  ⚠️ All models failed, last error: {last_error}")
    
    return best_fix, best_model

def is_valid_python(code: str) -> bool:
    """Basic Python syntax validation"""
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False

In [10]:
def save_fixed_code(filename: str, fixed_code: str) -> bool:
    """Enhanced code saver with validation"""
    filepath = FIXED_DIR / filename
    try:
        # Validate code before saving
        ast.parse(fixed_code)  # Syntax check
        with open(filepath, "w", encoding='utf-8') as f:
            f.write(fixed_code)
        return True
    except SyntaxError as e:
        print(f"❌ Invalid syntax in {filename}: {e}")
        return False
    except Exception as e:
        print(f"❌ File save error for {filename}: {e}")
        return False

def generate_repair_prompt(code: str, program_name: str = "") -> str:
    """Optimized prompt with algorithm-specific guidance"""
    algorithm_hints = {
        "bitcount": "Focus on bit manipulation operations",
        "breadth_first_search": "Check queue/visited node handling",
        "gcd": "Verify termination condition and return value"
    }
    
    hint = algorithm_hints.get(program_name.replace('.py',''), 
          "Check for off-by-one errors and boundary conditions")
    
    return f"""You are a Python repair expert. Fix ONE bug in this algorithm:

PROGRAM: {program_name}
HINT: {hint}
RULES:
1. Change ONLY the buggy line
2. Preserve all functionality
3. Maintain identical formatting

BUGGY CODE:
```python
{code}
"""







    example_text = ""
    if program_name in examples:
        example = examples[program_name]
        example_text = f"""
EXAMPLE FIX FOR THIS PROGRAM:
Buggy version: python {example['buggy']}

Fixed version: python {example['fixed']}

Reason: {example['explanation']}
"""

    # The indentation here needs to be consistent from the opening triple quotes
    return f"""
You are an expert Python debugger specializing in algorithmic implementations.

TASK: Fix the single-line bug in this Python program while preserving the original algorithm.

RULES:
1. There is EXACTLY ONE bug on ONE line
2. Preserve the original algorithm logic and structure
3. Make MINIMAL changes - change only what's necessary
4. Do NOT add comments or explanations
5. Return ONLY the corrected Python code
6. Maintain identical indentation and formatting
7. Ensure the fix passes all test cases

{example_text}

PROGRAM: {program_name}

BUGGY CODE: 
def bitcount(n):
    count = 0
    while n:
        n ^= n - 1
        count += 1
    return count

CORRECTED CODE: 
def bitcount(n):
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count 
"""

In [11]:
# 🟩 Cell 9: Enhanced Test Runner with Proper Syntax
from retrying import retry
from typing import Tuple

@retry(stop_max_attempt_number=3, wait_fixed=2000)
def run_test_with_validation(program_name: str) -> Tuple[str, int, bool]:
    """More robust test execution with proper error handling"""
    try:
        result = subprocess.run(
            ["python", "tester.py", program_name],
            capture_output=True,
            text=True,
            timeout=45,  # Increased timeout
            cwd=str(BASE_PATH)
        )
        output = (result.stdout + result.stderr).lower()
        passed = ("passed" in output) and (result.returncode == 0)
        
        # Debug info if failed
        if not passed:
            print(f"  🔍 Test Fail Details: {output[:300]}...")
            
        return output, 0 if passed else 1, passed
        
    except subprocess.TimeoutExpired:
        return "Timeout after 45 seconds", 1, False
    except Exception as e:
        return f"Test error: {str(e)}", 1, False





In [12]:
# 🟩 Cell 10: Complete Processing Cell with Similarity Calculation
from difflib import SequenceMatcher

def calculate_similarity(fixed_code: str, correct_path: Path) -> float:
    """Calculate code similarity score (0-1) with reference solution"""
    if not correct_path.exists():
        return 0.0
    
    with open(correct_path, 'r', encoding='utf-8') as f:
        correct_code = f.read()
    
    return SequenceMatcher(None, fixed_code, correct_code).ratio()

def is_valid_python(code: str) -> bool:
    """Validate Python syntax"""
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False

def process_program_comprehensive(filename: str) -> Optional[Dict]:
    print(f"\n📂 Processing: {filename}")
    
    # 1. Load and validate code
    original_code = load_code(BUGGY_DIR / filename)
    if not original_code or not is_valid_python(original_code):
        print(f"❌ Invalid code in {filename}")
        return None

    # 2. Initial test
    print("  🧪 Initial test...")
    output_before, _, passed_before = run_test_with_validation(filename)
    
    # 3. LLM Repair
    print("  🤖 Repairing...")
    fixed_code, model = fix_code_with_ollama(
        original_code, 
        filename,
        models=["codellama:7b-instruct", "codellama:13b-instruct"]
    )
    
    # 4. Validate and save
    if not is_valid_python(fixed_code):
        print(f"❌ Invalid fix generated for {filename}")
        return None
        
    if not save_fixed_code(filename, fixed_code):
        return None
        
    # 5. Test fix
    print("  🧪 Testing fix...")
    output_after, _, passed_after = run_test_with_validation(filename)
    
    # 6. Analysis
    defect_type = categorize_defect_advanced(original_code, fixed_code, filename)
    similarity = calculate_similarity(fixed_code, CORRECT_DIR / filename)
    
    result = {
        'filename': filename,
        'status': "PASS" if passed_after else "FAIL",
        'defect': defect_type,
        'similarity': round(similarity, 3),
        'model': model,
        'test_output': output_after[:500]  # Store first 500 chars of test output
    }
    
    print(f"  ✅ Result: {result['status']} | {defect_type} | Similarity: {similarity:.2f}")
    return result

In [13]:
# 🟩 Cell 10a: Parallel Batch Processing Implementation
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd

def batch_process_all_programs(max_workers: int = 4) -> List[Dict]:
    """Process all programs in parallel with caching and progress tracking"""
    print("🚀 Starting parallel batch processing...")
    
    # Get all Python files
    python_files = [f for f in os.listdir(BUGGY_DIR) if f.endswith(".py")]
    if not python_files:
        print(f"❌ No Python files found in {BUGGY_DIR}")
        return []
    
    print(f"📊 Found {len(python_files)} programs")
    
    # Initialize results and counters
    results = []
    success_count = 0
    
    # Process with thread pool
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Create future->filename mapping
        futures = {
            executor.submit(process_program_comprehensive, filename): filename
            for filename in python_files
        }
        
        # Process with progress bar
        for future in tqdm(as_completed(futures), 
                         total=len(python_files),
                         desc="Processing Programs"):
            filename = futures[future]
            try:
                result = future.result()
                if result:
                    results.append(result)
                    if result.get('repair_success', False):
                        success_count += 1
            except Exception as e:
                print(f"\n❌ Failed to process {filename}: {str(e)}")
    
    # Save raw results
    pd.DataFrame(results).to_csv(LOG_FILE, index=False)
    
    # Print summary
    print(f"\n{'='*50}")
    print(f"📊 Batch Processing Complete")
    print(f"• Total programs: {len(python_files)}")
    print(f"• Successful repairs: {success_count}")
    print(f"• Success rate: {success_count/len(python_files)*100:.1f}%")
    print(f"• Results saved to: {LOG_FILE}")
    print("="*50)
    
    return results

In [14]:
# 🟩 Cell 11: Execute the complete enhanced pipeline
def run_complete_pipeline():
    """Run the complete enhanced LLM agent pipeline"""
    print("🚀 Starting Enhanced QuixBugs LLM Agent Pipeline...")
    print(f"📁 Working with dataset at: {BASE_PATH}")
    
    # Step 1: Batch process all programs
    print("\n" + "="*60)
    print("STEP 1: PARALLEL BATCH PROCESSING")
    print("="*60)
    results = batch_process_all_programs()
    
    
    # Step 3: Export reports
    print("\n" + "="*60)
    print("STEP 3: EXPORT REPORTS")
    print("="*60)
    export_comprehensive_reports()
    
    # Step 4: Compare with benchmarks
    print("\n" + "="*60)
    print("STEP 4: BENCHMARK COMPARISON")
    print("="*60)
    compare_with_benchmarks()
    
    # Step 5: Failure analysis
    print("\n" + "="*60)
    print("STEP 5: FAILURE ANALYSIS")
    print("="*60)
    analyze_failures()
    
    print("\n" + "="*60)
    print("🎯 ENHANCED PIPELINE COMPLETE!")
    print("="*60)
    print("📊 Check the following files for results:")
    print("   • repair_detailed_log.csv - Raw data")
    print("   • quixbugs_repair_report.html - Interactive report")
    print("   • quixbugs_repair_summary.pdf - Summary report")
    print("   • comprehensive_analysis.png - Visualizations")
    print("   • repair_metrics.json - Metrics summary")
    print("   • benchmark_comparison.png - Benchmark results")
    print("   • failure_analysis.png - Failure patterns")
    
    return df

In [17]:
# 🟩 Cell 12: Main execution
if __name__ == "__main__":
    print("🚀 Enhanced QuixBugs LLM Agent - Ready to Execute!")
    print("="*60)
    print("Available functions:")
    print("1. run_complete_pipeline() - Run full enhanced analysis")
    print("2. test_single_program('filename.py') - Debug single program")
    print("3. compare_with_benchmarks() - Compare with existing tools")
    print("4. analyze_failures() - Analyze failed repairs")
    print("5. batch_process_all_programs() - Process all programs in parallel")
    
    print("="*60)
    
    # Uncomment to run the complete pipeline
    run_complete_pipeline()

🚀 Enhanced QuixBugs LLM Agent - Ready to Execute!
Available functions:
1. run_complete_pipeline() - Run full enhanced analysis
2. test_single_program('filename.py') - Debug single program
3. compare_with_benchmarks() - Compare with existing tools
4. analyze_failures() - Analyze failed repairs
5. batch_process_all_programs() - Process all programs in parallel
🚀 Starting Enhanced QuixBugs LLM Agent Pipeline...
📁 Working with dataset at: C:\Users\Rahul\Downloads\Code-Refactoring-QuixBugs-master\Code-Refactoring-QuixBugs-master

STEP 1: PARALLEL BATCH PROCESSING
🚀 Starting parallel batch processing...
📊 Found 50 programs

📂 Processing: bitcount.py
  🧪 Initial test...

📂 Processing: breadth_first_search.py

📂 Processing: breadth_first_search_test.py
  🧪 Initial test...
  🧪 Initial test...

📂 Processing: bucketsort.py
  🧪 Initial test...


Processing Programs:   0%|                                                                                   | 0/50 [00:00<?, ?it/s]

  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    print(f"\u274c test file not found for {algo}. skipping...")
  file "c:\users\rahul\downloads\anaconda\projects\lib\encodings\cp1...
  🤖 Repairing...
  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    print(f"\u274c test file not found for {algo}. skipping...")
  file "c:\users\rahul\downloads\anaconda\projects\lib\encodings\cp1...
  🤖 Repairing...
  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    print(f"\u274c test file not found for {algo}. skipping...")
  file "c:\users\rahul\downloads\anaconda\projects\lib\en

Processing Programs:   2%|█▌                                                                         | 1/50 [00:00<00:19,  2.55it/s]

  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    print(f"\u274c test file not found for {algo}. skipping...")
  file "c:\users\rahul\downloads\anaconda\projects\lib\encodings\cp1...
  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    print(f"\u274c test file not found for {algo}. skipping...")
  file "c:\users\rahul\downloads\anaconda\projects\lib\encodings\cp1...
  🧪 Initial test...
  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    print(f"\u274c test file not found for {algo}. skipping...")
  file "c:\users\rahul\downloads\anaconda\projects\lib\encodings\cp1...

...

Processing Programs:  96%|███████████████████████████████████████████████████████████████████████   | 48/50 [00:06<00:00,  6.94it/s]

  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    print(f"\u274c test file not found for {algo}. skipping...")
  file "c:\users\rahul\downloads\anaconda\projects\lib\encodings\cp1...
  🤖 Repairing...
  🔄 Using cached response for codellama:7b-instruct
  🧪 Testing fix...
  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    print(f"\u274c test file not found for {algo}. skipping...")
  file "c:\users\rahul\downloads\anaconda\projects\lib\encodings\cp1...
  ✅ Result: FAIL | missing_added_plus_one | Similarity: 0.06
  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    pri

Processing Programs: 100%|██████████████████████████████████████████████████████████████████████████| 50/50 [07:16<00:00,  8.74s/it]

  🧪 Testing fix...
  🔍 Test Fail Details: traceback (most recent call last):
  file "c:\users\rahul\downloads\code-refactoring-quixbugs-master\code-refactoring-quixbugs-master\tester.py", line 80, in <module>
    print(f"\u274c test file not found for {algo}. skipping...")
  file "c:\users\rahul\downloads\anaconda\projects\lib\encodings\cp1...
  ✅ Result: FAIL | incorrect_assignment_operator | Similarity: 0.47

📊 Batch Processing Complete
• Total programs: 50
• Successful repairs: 0
• Success rate: 0.0%
• Results saved to: repair_detailed_log.csv

STEP 2: COMPREHENSIVE ANALYSIS


NameError: name 'generate_comprehensive_analysis' is not defined